<a href="https://colab.research.google.com/github/mythien107/busad878/blob/main/Agentic_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Naive RAG

## Install dependencies

In [ ]:
!pip install -q anthropic sentence-transformers numpy

## Load the API key

In [ ]:
import os
from google.colab import userdata

# In Colab: click the key icon in the left sidebar and add a secret
# named ANTHROPIC_API_KEY. Then this loads it.
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

import anthropic
client = anthropic.Anthropic()

# Sanity check
print("API key loaded:", os.environ["ANTHROPIC_API_KEY"][:10] + "...")

## Sample document

In [ ]:
# Aurora Manufacturing's fictional PTO policy. We'll use this as our
# "knowledge base" — in production this might be 10,000 SharePoint pages.

POLICY_DOC = """
Aurora Manufacturing — Paid Time Off Policy (Effective January 2026)

1. Eligibility. All full-time U.S. employees are eligible for PTO from
their date of hire. Part-time employees who work at least 24 hours per
week accrue PTO at a prorated rate.

2. Accrual. Employees with less than 3 years of service accrue 15 days
of PTO per year. Employees with 3 to 7 years accrue 20 days. Employees
with more than 7 years of service accrue 25 days. Accrual is monthly,
and unused PTO may be carried over up to a maximum of 10 days.

3. Approval. PTO requests must be submitted at least 14 days in advance
through Workday. Requests for more than 5 consecutive days require
manager and HR approval. Requests during fiscal year-end close
(December 15 to January 5) are restricted to emergencies only.

4. Sick leave. Sick leave is separate from PTO. Employees receive 10
sick days per year, non-cumulative. Sick days do not require advance
notice but must be reported to the manager by 9:00 AM on the day of
absence.

5. Bereavement. Up to 5 days of paid bereavement leave is provided for
the loss of an immediate family member; up to 2 days for an extended
family member.

6. Parental leave. Aurora provides 16 weeks of paid parental leave for
the birth or adoption of a child, available to either parent.

7. Cash-out. Upon separation from the company, accrued and unused PTO
is paid out at the employee's final base salary rate, subject to
applicable taxes and a maximum of 30 days.
"""
print("Document loaded:", len(POLICY_DOC), "characters")

## Chunk the document

In [ ]:
chunks = [c.strip() for c in POLICY_DOC.split("\n\n") if c.strip()]
print(f"Number of chunks: {len(chunks)}")
print("\n--- First chunk ---\n", chunks[1][:200])

## Embed every chunk

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# A small, fast, free embedding model. Production systems use bigger
# models (OpenAI's text-embedding-3-large, Cohere, Voyage, etc.) but
# this is sufficient for a one-page document.
embedder = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = embedder.encode(chunks)
print("Embedding shape:", chunk_embeddings.shape)
# (8, 384) means 8 chunks, each represented by a 384-dimensional vector

## The retrieval function

In [ ]:
def retrieve(query: str, k: int = 3):
    """Return the top-k chunks most similar to the query."""
    query_embedding = embedder.encode([query])[0]

    # Cosine similarity: how aligned are the two vectors?
    similarities = chunk_embeddings @ query_embedding / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )

    top_indices = np.argsort(similarities)[::-1][:k]
    return [(chunks[i], float(similarities[i])) for i in top_indices]

# Try it
question = "How much PTO do I get if I've been here for five years?"
results = retrieve(question)
for chunk, score in results:
    print(f"[score={score:.3f}]")
    print(chunk[:150])
    print("---")

## Combine retrieval with generation

In [ ]:
def ask(question: str, k: int = 3):
    """Retrieve relevant chunks, then ask Claude to answer using them."""
    retrieved = retrieve(question, k=k)
    context = "\n\n".join([chunk for chunk, _ in retrieved])

    prompt = f"""You are an HR assistant for Aurora Manufacturing.
Answer the user's question using ONLY the policy excerpts below.
If the answer is not in the excerpts, say "I don't have that information
in the policy. Please contact HR."

POLICY EXCERPTS:
{context}

USER QUESTION: {question}
"""

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

# Run it
answer = ask("How much PTO do I get if I've been here for five years?")
print(answer)

## Failure Mode

In [ ]:
# Ask a question whose answer is NOT in the policy.
print(ask("What's Aurora's policy on remote work from international locations?"))

In [ ]:
# Same question, but we REMOVE the "if not in excerpts, say I don't know" guardrail.
def ask_unsafe(question: str, k: int = 3):
    retrieved = retrieve(question, k=k)
    context = "\n\n".join([chunk for chunk, _ in retrieved])
    prompt = f"""Answer the user's question using the context.

CONTEXT: {context}

QUESTION: {question}
"""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text

print(ask_unsafe("What's Aurora's policy on remote work from international locations?"))

# Add Self-Critique

## A self-critique function

In [ ]:
import json

def critique(question: str, context: str, answer: str) -> dict:
    """Have Claude evaluate its own answer for faithfulness and completeness."""
    prompt = f"""You are evaluating a RAG system's output. Be strict.

CONTEXT GIVEN TO THE ASSISTANT:
{context}

USER QUESTION: {question}

ASSISTANT'S ANSWER: {answer}

Evaluate the answer on two dimensions, score 0-10:
1. faithfulness: are all claims in the answer supported by the context?
2. completeness: does the answer fully address the question?

Also decide whether the system should retry with a different query.
Reply ONLY with valid JSON in this exact format:
{{"faithfulness": <int>, "completeness": <int>, "should_retry": <bool>, "reason": "<brief>"}}
"""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )
    return json.loads(response.content[0].text)

## A query rewriter

In [ ]:
def rewrite_query(original_question: str, last_answer: str, last_critique: dict) -> str:
    """If the answer was bad, rewrite the question to retrieve better chunks."""
    prompt = f"""The user asked: "{original_question}"
The system answered: "{last_answer}"
Evaluation said: {last_critique['reason']}

Suggest ONE alternative search query that might retrieve more useful information
from a corporate policy database. Reply with just the query, nothing else."""
    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=100,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.content[0].text.strip().strip('"')

## The agentic loop

In [ ]:
def agentic_ask(question: str, max_attempts: int = 3):
    current_query = question
    history = []

    for attempt in range(max_attempts):
        print(f"\n=== Attempt {attempt+1}: '{current_query}' ===")

        retrieved = retrieve(current_query, k=3)
        context = "\n\n".join([chunk for chunk, _ in retrieved])

        # Same generation prompt as Tutorial 1
        prompt = f"""You are an HR assistant for Aurora Manufacturing.
Answer using ONLY the policy excerpts. If unanswerable, say so.

POLICY EXCERPTS:
{context}

USER QUESTION: {question}"""
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=400,
            messages=[{"role": "user", "content": prompt}],
        )
        answer = response.content[0].text

        # Self-critique
        review = critique(question, context, answer)
        print(f"Faithfulness: {review['faithfulness']}/10")
        print(f"Completeness: {review['completeness']}/10")
        print(f"Should retry: {review['should_retry']}")

        history.append({"query": current_query, "answer": answer, "review": review})

        if not review["should_retry"] or review["completeness"] >= 8:
            return answer, history

        # Rewrite for next loop
        current_query = rewrite_query(question, answer, review)

    return answer, history

# Try a question that requires combining info from multiple sections
final_answer, trace = agentic_ask(
    "If I take 3 weeks off in December, can I cash out the rest of my PTO when I leave?"
)
print("\n\n=== FINAL ANSWER ===\n", final_answer)

# Tool Use Agent (ReAct)

## A second knowledge base + a calculator

In [ ]:
# A second corpus: fictional Q3 2026 financial summary
FINANCIALS = """
Aurora Manufacturing — Q3 2026 Financial Summary

Revenue: $487M (Q3 2025: $412M)
Gross profit: $185M (Q3 2025: $148M)
Operating income: $62M (Q3 2025: $44M)
Net income: $48M (Q3 2025: $33M)

Headcount at quarter end: 4,120 employees
Capital expenditure in Q3: $28M, primarily on the Toledo facility expansion.

Forward guidance: Q4 revenue expected $510M-$525M, full-year revenue
expected $1.92B-$1.94B.
"""

policy_chunks = chunks  # from Tutorial 1
policy_emb = chunk_embeddings  # from Tutorial 1

financial_chunks = [c.strip() for c in FINANCIALS.split("\n\n") if c.strip()]
financial_emb = embedder.encode(financial_chunks)

def search_policy(query: str) -> str:
    sims = policy_emb @ embedder.encode([query])[0] / (
        np.linalg.norm(policy_emb, axis=1) * np.linalg.norm(embedder.encode([query])[0])
    )
    top = np.argsort(sims)[::-1][:2]
    return "\n---\n".join([policy_chunks[i] for i in top])

def search_financials(query: str) -> str:
    sims = financial_emb @ embedder.encode([query])[0] / (
        np.linalg.norm(financial_emb, axis=1) * np.linalg.norm(embedder.encode([query])[0])
    )
    top = np.argsort(sims)[::-1][:2]
    return "\n---\n".join([financial_chunks[i] for i in top])

def calculator(expression: str) -> str:
    # In production, use a sandboxed evaluator. For demo, eval is fine.
    try:
        result = eval(expression, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

print("Three tools defined: search_policy, search_financials, calculator")

## Use Anthropic's built-in tool use

In [ ]:
# Define the tools in the format Claude expects
tools = [
    {
        "name": "search_policy",
        "description": "Search Aurora's HR policy documents (PTO, benefits, leave). Use for any question about employee policies.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
    {
        "name": "search_financials",
        "description": "Search Aurora's quarterly financial summaries. Use for revenue, profit, headcount, capex questions.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
    {
        "name": "calculator",
        "description": "Evaluate a numerical expression. Example: '0.15 * 487'.",
        "input_schema": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
        },
    },
]

def run_tool(name: str, args: dict) -> str:
    if name == "search_policy":
        return search_policy(args["query"])
    if name == "search_financials":
        return search_financials(args["query"])
    if name == "calculator":
        return calculator(args["expression"])
    return f"Unknown tool: {name}"

## The agent loop

In [ ]:
def agent(user_question: str, max_steps: int = 6):
    messages = [{"role": "user", "content": user_question}]

    for step in range(max_steps):
        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        # If the model decided to answer, we're done
        if response.stop_reason == "end_turn":
            final = next(b.text for b in response.content if hasattr(b, "text"))
            print(f"\n=== FINAL ANSWER ===\n{final}")
            return final

        # Otherwise it's calling a tool — execute and feed result back
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"\nStep {step+1}: calling {block.name}({block.input})")
                result = run_tool(block.name, block.input)
                print(f"   → {result[:120]}{'...' if len(result) > 120 else ''}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                })
        messages.append({"role": "user", "content": tool_results})

    return "Max steps reached without a final answer."

## Ask a question that requires all three tools

In [ ]:
agent("""
Our CEO wants to know: if we set aside 15% of Q3 operating income as a
discretionary employee bonus pool, and we split it equally across all
employees, how much does each person get? Also, are bonuses subject to
the same approval workflow as PTO requests?
""")

# Evaluate with Ragas Metrics

## A small evaluation set

In [ ]:
# A tiny eval set. In production this should be 50-500 questions, ideally
# written by domain experts who know the right answers.
eval_set = [
    {
        "question": "How many PTO days do I get if I've worked here for two years?",
        "ground_truth": "15 days, since you have less than 3 years of service.",
    },
    {
        "question": "Can I take PTO during fiscal year-end close?",
        "ground_truth": "Only for emergencies between December 15 and January 5.",
    },
    {
        "question": "What's the parental leave policy?",
        "ground_truth": "16 weeks paid, available to either parent.",
    },
    {
        "question": "What's Aurora's policy on company cars?",
        "ground_truth": "Not in the policy document. Should refuse.",
    },
]

### 'Faithfulness' metric

In [ ]:
def faithfulness_score(question: str, context: str, answer: str) -> float:
    """Fraction of claims in the answer that are supported by the context."""
    # Step 1: extract atomic claims from the answer
    extract = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        messages=[{"role": "user", "content":
            f"List each factual claim in the answer below as a numbered list, "
            f"one short claim per line. Answer:\n{answer}"
        }],
    )
    claims = [c.strip() for c in extract.content[0].text.split("\n") if c.strip() and c.strip()[0].isdigit()]

    if not claims:
        return 1.0

    # Step 2: check each claim against context
    supported = 0
    for claim in claims:
        check = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=10,
            messages=[{"role": "user", "content":
                f"Context:\n{context}\n\nClaim: {claim}\n\n"
                f"Is the claim supported by the context? Reply YES or NO only."
            }],
        )
        if "YES" in check.content[0].text.upper():
            supported += 1

    return supported / len(claims)

## 'Answer relevance' metric

In [ ]:
def answer_relevance_score(question: str, answer: str) -> float:
    """Generate plausible questions from the answer; measure similarity to the original."""
    gen = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=300,
        messages=[{"role": "user", "content":
            f"Given this answer, generate 3 questions it would plausibly answer. "
            f"One per line, no numbering.\nAnswer: {answer}"
        }],
    )
    generated = [q.strip() for q in gen.content[0].text.split("\n") if q.strip()]
    if not generated:
        return 0.0

    q_emb = embedder.encode([question])[0]
    g_embs = embedder.encode(generated)
    sims = g_embs @ q_emb / (np.linalg.norm(g_embs, axis=1) * np.linalg.norm(q_emb))
    return float(np.mean(sims))

## Run the eval

In [ ]:
print(f"{'Question':<55} | {'Faith':<6} | {'Rel':<6}")
print("-" * 80)

for case in eval_set:
    retrieved = retrieve(case["question"], k=3)
    context = "\n\n".join([chunk for chunk, _ in retrieved])
    answer = ask(case["question"])

    faith = faithfulness_score(case["question"], context, answer)
    rel = answer_relevance_score(case["question"], answer)

    print(f"{case['question'][:54]:<55} | {faith:<6.2f} | {rel:<6.2f}")

In production, just use the real library:
```
!pip install ragas
from ragas.metrics import faithfulness, answer_relevance, context_precision
```
It will run dozens of metrics, integrate with LangChain & LlamaIndex, and produce dashboards you can show your CIO.